# Установка зависимостей

In [46]:
%pip install torch torchvision onnx onnxruntime numpy


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# Импорты

In [47]:
import os
import json
import numpy as np

import torch
import torch.nn as nn
from torchvision.models import efficientnet_b0

import onnx
import onnxruntime as ort

# Настройки

In [48]:
CHECKPOINT_PATH = "best.pth"
ONNX_OUTPUT_PATH = "bird_model.onnx"

CLASSES_PATH = "classes.json"

BATCH_SIZE = 1
CHANNELS = 3
N_MELS = 128
N_FRAMES = 501

INPUT_SHAPE = (BATCH_SIZE, CHANNELS, N_MELS, N_FRAMES)


OPSET_VERSION = 17

DEVICE = "cpu"

# Загрузка Checkpoint

In [49]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)

print(type(checkpoint))

if isinstance(checkpoint, dict):
    for key in checkpoint.keys():
        print(" -", key)

<class 'dict'>
 - model
 - classes
 - label2idx
 - config


# Загрузка классов из class.json

In [50]:
with open(CLASSES_PATH, "r", encoding="utf-8") as f:
    loaded_classes = json.load(f)

print(type(loaded_classes))

if isinstance(loaded_classes, list):
    classes = loaded_classes

<class 'list'>


In [51]:
for i, class_name in enumerate(classes):
    print(i, class_name)

0 anas_platyrhynchos
1 ardea_cinerea
2 carduelis_carduelis
3 coloeus_monedula
4 columba_palumbus
5 corvus_frugilegus
6 fringilla_coelebs
7 hirundo_rustica
8 luscinia_luscinia
9 luscinia_svecica
10 motacilla_alba
11 parus_major
12 pica_pica
13 strix_aluco
14 turdus_merula
15 turdus_ruficollis


# Создание модели EfficientNet

In [52]:
def create_model(num_classes: int):
    model = efficientnet_b0(weights=None)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model

num_classes = len(classes)
model = create_model(num_classes).to(DEVICE)

print(model)

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

# Загрузка весов в модель

In [53]:
state_dict = checkpoint["model"]
missing_keys, unexpected_keys = model.load_state_dict(state_dict, strict=False)
print(missing_keys, unexpected_keys)
model.eval()

[] []


EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

# Тестовый вход

In [54]:
dummy_input = torch.randn(INPUT_SHAPE, dtype=torch.float32).to(DEVICE)

print(dummy_input.shape)

torch.Size([1, 3, 128, 501])


In [55]:
with torch.no_grad():
    torch_output = model(dummy_input)

print(torch_output.shape)
print(torch_output[0])

torch.Size([1, 16])
tensor([ -901.9171, -2056.9036,  -759.0696, -1626.1921, -1207.2135,  -460.7949,
         -236.5254, -2032.6766,  -807.6553,  -438.8158, -2560.9360,  1768.3778,
         -317.8059, -1811.7883,   856.4803, -1640.3387])


# экспорт в ONNX

In [56]:
model.eval()
torch.onnx.export(
    model,
    dummy_input,
    ONNX_OUTPUT_PATH,
    export_params=True,
    opset_version=OPSET_VERSION,
    do_constant_folding=True,
    input_names=["input"],
    output_names=["logits"],
    dynamic_axes={
        "input": {0: "batch_size"},
        "logits": {0: "batch_size"},
    }
)

ONNX_OUTPUT_PATH

'bird_model.onnx'

In [57]:
session = ort.InferenceSession(ONNX_OUTPUT_PATH, providers=["CPUExecutionProvider"])

print("Входы ONNX:")
for inp in session.get_inputs():
    print("name:", inp.name)
    print("shape:", inp.shape)
    print("type:", inp.type)
    print()

print("Выходы ONNX:")
for out in session.get_outputs():
    print("name:", out.name)
    print("shape:", out.shape)
    print("type:", out.type)
    print()

Входы ONNX:
name: input
shape: ['batch_size', 3, 128, 501]
type: tensor(float)

Выходы ONNX:
name: logits
shape: ['batch_size', 16]
type: tensor(float)



In [ ]:
model.eval()

with torch.no_grad():
    torch_output = model(dummy_input)
input_numpy = dummy_input.cpu().numpy()

onnx_outputs = session.run(
    ["logits"],
    {
        "input": input_numpy
    }
)

onnx_output = onnx_outputs[0]

torch_output_numpy = torch_output.detach().cpu().numpy()

max_abs_diff = np.max(np.abs(onnx_output - torch_output_numpy))
mean_abs_diff = np.mean(np.abs(onnx_output - torch_output_numpy))

print("Максимальная абсолютная разница:", max_abs_diff)
print("Средняя абсолютная разница:", mean_abs_diff)

torch_output_numpy[0], onnx_output[0]

Максимальная абсолютная разница: 1.0944824
Средняя абсолютная разница: 0.47449398


(array([ -901.9171 , -2056.9036 ,  -759.0696 , -1626.1921 , -1207.2135 ,
         -460.79492,  -236.52539, -2032.6766 ,  -807.65533,  -438.81583,
        -2560.936  ,  1768.3778 ,  -317.80588, -1811.7883 ,   856.4803 ,
        -1640.3387 ], dtype=float32),
 array([ -901.58185, -2056.161  ,  -758.72424, -1625.6323 , -1206.863  ,
         -460.56744,  -236.4312 , -2031.9568 ,  -807.2355 ,  -438.60263,
        -2559.8416 ,  1767.6003 ,  -317.8366 , -1811.081  ,   856.2141 ,
        -1639.6311 ], dtype=float32))

In [61]:
def softmax(x):
    x = x - np.max(x)
    e_x = np.exp(x)
    return e_x / e_x.sum()

probs = softmax(onnx_output[0])

top_k = 3
top_indices = np.argsort(probs)[::-1][:top_k]

for idx in top_indices:
    print(f"{classes[idx]}: {probs[idx] * 100:.2f}%")

parus_major: 100.00%
turdus_ruficollis: 0.00%
strix_aluco: 0.00%


In [62]:
probs = softmax(torch_output_numpy[0])

top_k = 3
top_indices = np.argsort(probs)[::-1][:top_k]

for idx in top_indices:
    print(f"{classes[idx]}: {probs[idx] * 100:.2f}%")

parus_major: 100.00%
turdus_ruficollis: 0.00%
strix_aluco: 0.00%


In [63]:
import time

num_runs = 50

for _ in range(5):
    _ = session.run(["logits"], {"input": input_numpy})

times = []

for _ in range(num_runs):
    start = time.perf_counter()
    _ = session.run(["logits"], {"input": input_numpy})
    end = time.perf_counter()

    times.append((end - start) * 1000)

times = np.array(times)
print(f"Среднее время выполнения: {np.mean(times)} ms")
print(f"Медианное время выполнения: {np.median(times)} ms")
print(f"Минимальное время выполнения: {np.min(times)} ms")
print(f"Максимальное время выполнения: {np.max(times)} ms")
print(f"Среднее время выполнения: {np.mean(times)} ms")

Среднее время выполнения: 8.194375280108943 ms
Медианное время выполнения: 8.067102499808243 ms
Минимальное время выполнения: 7.653722999748425 ms
Максимальное время выполнения: 9.359505000247736 ms
Среднее время выполнения: 8.194375280108943 ms
